### Purpose

1. 약품 부분 수정 · 별칭 · 이력 API 배포 (docs/gdb-update-api-request.md 4장)
2. **update/alias 쓰기 API에 API 키(`x-api-key`) 적용** (요청서 3장) — 거래처·약품 4개 경로

| 대상 | 내용 |
|---|---|
| DynamoDB `dschemical_history` | 키 `dsids` + `at` |
| Lambda `dsglobaldbCreate` (갱신) | `/dschemical/update`·`alias`·`history` 추가 (거래처와 같은 코드) |
| 사용량 계획 `globaldb-write` | 스테이지 `dev` 연결, 앱별 키: `globaldb-inv`(재고), `globaldb-quote`(견적), `globaldb-web`(웹) |
| 키 필요 | `POST /dscustomer/update`, `/dscustomer/alias`, `/dschemical/update`, `/dschemical/alias` |

- 키가 없어도 되는 것: 모든 GET(조회·이력), create API, 기존 `/dschemical` `/dscustomer` `/dswarehouse` 의 PUT/POST/DELETE (웹 엑셀 비교용)
- 키 값은 **출력하지 않고** 아래 파일·설정에만 넣는다 (모두 git 제외)
  - `AWS/Lambda/.env` : 재고·견적 쪽에 전달할 키
  - 프로젝트 루트 `.env.local` : 웹 개발 서버용 `VITE_GDB_WEB_API_KEY`
  - Amplify 앱 `dsglobaldb_manager_web` 환경 변수 `VITE_GDB_WEB_API_KEY` : 배포 빌드용
- 웹 키는 브라우저 코드에 들어가므로 비밀이 아니다. 키를 앱별로 나눈 목적은 앱 구분과 앱별 차단(키 비활성화)이다
- **마지막 셀(`dev` 스테이지 배포)은 API 전체를 다시 배포한다**

In [ ]:
# Norton 등이 HTTPS를 가로채는 PC에서 SSL 오류 방지 - boto3 보다 먼저 적용해야 함
import truststore
truststore.inject_into_ssl()

import json
import urllib.request
from pathlib import Path

import boto3

boto3.setup_default_session(profile_name='default', region_name='us-east-1')
from deploy_helpers import (deploy_lambda, ensure_resource_path, get_request_template, setup_lambda_method,
                            setup_options, grant_invoke, ensure_api_keys, deploy_stage_and_check)

Settings

In [ ]:
HISTORY_TABLE = 'dschemical_history'
ROLE_ARN = 'arn:aws:iam::199525800117:role/DSmapLambda'

API_ID = 'jyipsj28s9'
STAGE = 'dev'
TEMPLATE_FROM = '/dschemical'
FUNCTION_NAME = 'dsglobaldbCreate'
SOURCE_FILE = './dsglobaldbCreate/lambda_function.py'

# 경로 → [(메서드, API 키 필요)]
ROUTES = {
    'dscustomer/update': [('POST', True)],
    'dscustomer/alias': [('POST', True)],
    'dscustomer/history': [('GET', False)],
    'dschemical/update': [('POST', True)],
    'dschemical/alias': [('POST', True)],
    'dschemical/history': [('GET', False)],
}
PLAN_NAME = 'globaldb-write'
KEY_NAMES = {'globaldb-inv': 'GDB_API_KEY_INV', 'globaldb-quote': 'GDB_API_KEY_QUOTE', 'globaldb-web': 'GDB_API_KEY_WEB'}
AMPLIFY_APP = 'dsglobaldb_manager_web'

PROJECT_ROOT = Path('..', '..').resolve()
ACCOUNT_ID = ROLE_ARN.split(':')[4]

1. 약품 변경 이력 테이블

In [ ]:
ddb = boto3.client('dynamodb')
if HISTORY_TABLE in ddb.list_tables()['TableNames']:
    print('테이블 있음:', HISTORY_TABLE)
else:
    ddb.create_table(
        TableName=HISTORY_TABLE,
        KeySchema=[{'AttributeName': 'dsids', 'KeyType': 'HASH'}, {'AttributeName': 'at', 'KeyType': 'RANGE'}],
        AttributeDefinitions=[{'AttributeName': 'dsids', 'AttributeType': 'S'}, {'AttributeName': 'at', 'AttributeType': 'S'}],
        BillingMode='PAY_PER_REQUEST',
    )
    ddb.get_waiter('table_exists').wait(TableName=HISTORY_TABLE)
    print('테이블 생성:', HISTORY_TABLE)

2. Lambda 갱신 + 경로 (update/alias 는 API 키 필요)

In [ ]:
function_arn = deploy_lambda(FUNCTION_NAME, SOURCE_FILE, ROLE_ARN,
                             description='글로벌 DB 안전한 쓰기 (create · 거래처/약품 update/alias/history)', timeout=29, memory=256)
request_template = get_request_template(API_ID, TEMPLATE_FROM)
for path, methods in ROUTES.items():
    resource_id = ensure_resource_path(API_ID, path)
    for method, key_required in methods:
        setup_lambda_method(API_ID, resource_id, method, function_arn, request_template, api_key_required=key_required)
    setup_options(API_ID, resource_id, [m for m, _ in methods])
    grant_invoke(FUNCTION_NAME, API_ID, ACCOUNT_ID, path)

3. 사용량 계획 + 앱별 API 키 → 로컬 파일·Amplify 환경 변수 (값은 출력 안 함)

In [ ]:
keys = ensure_api_keys(API_ID, STAGE, PLAN_NAME, list(KEY_NAMES))

env_path = Path('.env')     # AWS/Lambda/.env (git 제외)
lines = [l for l in (env_path.read_text(encoding='utf-8').splitlines() if env_path.exists() else [])
         if not l.startswith(tuple(KEY_NAMES.values()))]
lines += [f'{KEY_NAMES[name]}={value}' for name, value in keys.items()]
env_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print('저장:', env_path.resolve())

web_env = PROJECT_ROOT / '.env.local'   # vite 개발 서버용 (git 제외: *.local)
web_lines = [l for l in (web_env.read_text(encoding='utf-8').splitlines() if web_env.exists() else [])
             if not l.startswith('VITE_GDB_WEB_API_KEY=')]
web_env.write_text('\n'.join(web_lines + [f"VITE_GDB_WEB_API_KEY={keys['globaldb-web']}"]) + '\n', encoding='utf-8')
print('저장:', web_env)

amplify = boto3.client('amplify')
app = next(a for a in amplify.list_apps()['apps'] if a['name'] == AMPLIFY_APP)
env_vars = dict(app.get('environmentVariables') or {})
env_vars['VITE_GDB_WEB_API_KEY'] = keys['globaldb-web']
amplify.update_app(appId=app['appId'], environmentVariables=env_vars)
print('Amplify 환경 변수 설정:', AMPLIFY_APP, sorted(env_vars))

4. `dev` 스테이지 배포 + 확인 (쓰기 없음: 키 없으면 403, 키 있으면 Lambda 까지 가서 입력 오류 400)

In [ ]:
deploy_stage_and_check(API_ID, STAGE, 'dschemical/history?id=__check__',
                       'add /dschemical/update, alias, history; API key on update/alias')

url = f'https://{API_ID}.execute-api.us-east-1.amazonaws.com/{STAGE}/dscustomer/update'
def post(headers):
    req = urllib.request.Request(url, method='POST', data=b'{}', headers={'Content-Type': 'application/json', **headers})
    try:
        with urllib.request.urlopen(req) as r:
            return r.status, json.load(r).get('statusCode')
    except urllib.error.HTTPError as e:
        return e.code, None
print('키 없음  :', post({}))
print('키 있음  :', post({'x-api-key': keys['globaldb-inv']}))